In [1]:
import scanpy as sc
import dynamo as dyn
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

/home/liangzhen/anaconda3/envs/cellrank2/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/home/liangzhen/anaconda3/envs/cellrank2/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/home/liangzhen/anaconda3/envs/cellrank2/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for

In [2]:
dyn.__version__

'1.4.0'

In [3]:
color_map = {
    '1':'#a6cee3',
    '2':'#54278f',
    '3':'#e7298a',
    '4':'#1f77b4',
    '5':'#ff6600'
}

In [4]:
metadata = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/scRNA/cellranger/scRNA.metadata.csv',header=0,index_col=0)
metadata.index = metadata['cellName'].to_list()

/tmp/ipykernel_47633/651495206.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/scRNA/cellranger/scRNA.metadata.csv',header=0,index_col=0)


In [5]:
adata = sc.read_h5ad('/syn1/liangzhen/jinhua_jilab_project/result/scRNA/h5ad/adata.h5ad')
adata = adata[adata.obs.cellname.isin(metadata.cellName),]
adata.obs['lineageGrp'] = metadata.loc[adata.obs.cellname,'lineageGrp'].to_list()
adata.obsm['X_umap'] = np.array(metadata.loc[adata.obs.cellname,['umapharmony_1','umapharmony_2']])
adata.obs['cluster'] = metadata.loc[adata.obs.cellname,'tumor_state'].to_list()
adata.obs['cluster'] = adata.obs['cluster'].astype('str')
adata.obs['time'] = metadata.loc[adata.obs.cellname,'time'].to_list()

/tmp/ipykernel_47633/2811812953.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['lineageGrp'] = metadata.loc[adata.obs.cellname,'lineageGrp'].to_list()


# BL

In [6]:
transition_files = os.listdir('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/BL/')
transition_files = [ filename for filename in transition_files if filename.startswith('C')]
transition_files

['C59_state_transition.csv',
 'C42_state_transition.csv',
 'C15_state_transition.csv',
 'C75_state_transition.csv',
 'C83_state_transition.csv',
 'C29_state_transition.csv',
 'C12_state_transition.csv',
 'C32_state_transition.csv',
 'C60_state_transition.csv',
 'C90_state_transition.csv',
 'C20_state_transition.csv',
 'C38_state_transition.csv',
 'C73_state_transition.csv',
 'C43_state_transition.csv',
 'C89_state_transition.csv',
 'C63_state_transition.csv',
 'C26_state_transition.csv',
 'C96_state_transition.csv',
 'C2_state_transition.csv',
 'C53_state_transition.csv',
 'C3_state_transition.csv',
 'C8_state_transition.csv',
 'C14_state_transition.csv',
 'C19_state_transition.csv',
 'C18_state_transition.csv',
 'C7_state_transition.csv',
 'C22_state_transition.csv',
 'C36_state_transition.csv',
 'C65_state_transition.csv',
 'C13_state_transition.csv',
 'C79_state_transition.csv',
 'C66_state_transition.csv',
 'C95_state_transition.csv',
 'C21_state_transition.csv',
 'C45_state_transi

In [14]:
for transition_file in transition_files:
    print(transition_file)
    transition = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/BL/'+transition_file,index_col=0)
    transition.index = transition.index.astype('str')
    transition.columns = transition.index
    lineageGrp = transition_file.split('_')[0]
    clusters = transition.index.astype('str')
    adata_selected = adata[(adata.obs['lineageGrp']==lineageGrp) & 
                           (adata.obs['cluster'].isin(clusters)) &
                           (adata.obs['time'].isin(['T1'])) 
                          ]
    colors = [color_map[cluster] for cluster in transition.index]
    transition = transition.loc[adata_selected.obs['cluster'].unique(),adata_selected.obs['cluster'].unique()]
    adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])
    
    dyn.pl.state_graph(adata_selected,pointsize=1.5,facecolor='#000000',edgecolor='#000000',#edge_scale=15,
                   color=['cluster'],
                   group='cluster',
                   basis='umap',
                   state_graph=transition.to_numpy(),
                   #method='vf',
                   figsize=[4,4],
                   color_key=colors,#frontier =True
                   save_show_or_return='save',
                   save_kwargs = {'path': '/syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/BL/', 'prefix': lineageGrp,'ext': 'svg',
                                 "transparent": True, "close": True, "verbose": True}
                  )
    #plt.title(transition_file)
    #break

C59_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/BL/C59_dyn_savefig.svg...


/tmp/ipykernel_47633/2569517099.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C42_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/BL/C42_dyn_savefig.svg...


/tmp/ipykernel_47633/2569517099.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C15_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/BL/C15_dyn_savefig.svg...


...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

# EE

In [15]:
transition_files = os.listdir('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/EE/')
transition_files = [ filename for filename in transition_files if filename.startswith('C')]
transition_files

['C49_state_transition.csv',
 'C23_state_transition.csv',
 'C15_state_transition.csv',
 'C30_state_transition.csv',
 'C12_state_transition.csv',
 'C20_state_transition.csv',
 'C27_state_transition.csv',
 'C2_state_transition.csv',
 'C3_state_transition.csv',
 'C17_state_transition.csv',
 'C8_state_transition.csv',
 'C14_state_transition.csv',
 'C18_state_transition.csv',
 'C25_state_transition.csv',
 'C22_state_transition.csv',
 'C13_state_transition.csv',
 'C10_state_transition.csv',
 'C5_state_transition.csv',
 'C6_state_transition.csv',
 'C39_state_transition.csv',
 'C33_state_transition.csv',
 'C9_state_transition.csv',
 'C1_state_transition.csv']

In [16]:
for transition_file in transition_files:
    print(transition_file)
    transition = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/EE/'+transition_file,index_col=0)
    transition.index = transition.index.astype('str')
    transition.columns = transition.index
    lineageGrp = transition_file.split('_')[0]
    clusters = transition.index.astype('str')
    adata_selected = adata[(adata.obs['lineageGrp']==lineageGrp) & 
                           (adata.obs['cluster'].isin(clusters)) &
                           (adata.obs['time'].isin(['T2'])) 
                          ]
    colors = [color_map[cluster] for cluster in transition.index]
    transition = transition.loc[adata_selected.obs['cluster'].unique(),adata_selected.obs['cluster'].unique()]
    adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])
    
    dyn.pl.state_graph(adata_selected,pointsize=1.5,facecolor='#000000',edgecolor='#000000',#edge_scale=15,
                   color=['cluster'],
                   group='cluster',
                   basis='umap',
                   state_graph=transition.to_numpy(),
                   #method='vf',
                   figsize=[4,4],
                   color_key=colors,#frontier =True
                   save_show_or_return='save',
                   save_kwargs = {'path': '/syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/EE/', 'prefix': lineageGrp,'ext': 'svg',
                                 "transparent": True, "close": True, "verbose": True}
                  )
    #plt.title(transition_file)
    #break

C49_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/EE/C49_dyn_savefig.svg...


/tmp/ipykernel_47633/2603650982.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C23_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/EE/C23_dyn_savefig.svg...


/tmp/ipykernel_47633/2603650982.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C15_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/EE/C15_dyn_savefig.svg...


...

Done
C9_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/EE/C9_dyn_savefig.svg...


/tmp/ipykernel_47633/2603650982.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C1_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type


/tmp/ipykernel_47633/2603650982.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/EE/C1_dyn_savefig.svg...
Done


# LE

In [21]:
transition_files = os.listdir('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/LE/')
transition_files = [ filename for filename in transition_files if filename.startswith('C')]
transition_files

['C35_state_transition.csv',
 'C23_state_transition.csv',
 'C15_state_transition.csv',
 'C48_state_transition.csv',
 'C12_state_transition.csv',
 'C11_state_transition.csv',
 'C20_state_transition.csv',
 'C27_state_transition.csv',
 'C58_state_transition.csv',
 'C31_state_transition.csv',
 'C24_state_transition.csv',
 'C2_state_transition.csv',
 'C3_state_transition.csv',
 'C17_state_transition.csv',
 'C8_state_transition.csv',
 'C14_state_transition.csv',
 'C18_state_transition.csv',
 'C25_state_transition.csv',
 'C13_state_transition.csv',
 'C21_state_transition.csv',
 'C10_state_transition.csv',
 'C40_state_transition.csv',
 'C5_state_transition.csv',
 'C28_state_transition.csv',
 'C6_state_transition.csv',
 'C41_state_transition.csv',
 'C33_state_transition.csv',
 'C9_state_transition.csv',
 'C1_state_transition.csv']

In [22]:
for transition_file in transition_files:
    print(transition_file)
    transition = pd.read_csv('/syn1/liangzhen/jinhua_jilab_project/result/DNA_Amplicon/cell_transition/LE/'+transition_file,index_col=0)
    transition.index = transition.index.astype('str')
    transition.columns = transition.index
    lineageGrp = transition_file.split('_')[0]
    clusters = transition.index.astype('str')
    adata_selected = adata[(adata.obs['lineageGrp']==lineageGrp) & 
                           (adata.obs['cluster'].isin(clusters)) &
                           (adata.obs['time'].isin(['T3'])) 
                          ]
    colors = [color_map[cluster] for cluster in transition.index]
    transition = transition.loc[adata_selected.obs['cluster'].unique(),adata_selected.obs['cluster'].unique()]
    adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])
    
    dyn.pl.state_graph(adata_selected,pointsize=1.5,facecolor='#000000',edgecolor='#000000',#edge_scale=15,
                   color=['cluster'],
                   group='cluster',
                   basis='umap',
                   state_graph=transition.to_numpy(),
                   #method='vf',
                   figsize=[4,4],
                   color_key=colors,#frontier =True
                   save_show_or_return='save',
                   save_kwargs = {'path': '/syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/LE/', 'prefix': lineageGrp,'ext': 'svg',
                                 "transparent": True, "close": True, "verbose": True}
                  )
    #plt.title(transition_file)
    #break

C35_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/LE/C35_dyn_savefig.svg...


/tmp/ipykernel_47633/2587552383.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C23_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/LE/C23_dyn_savefig.svg...


/tmp/ipykernel_47633/2587552383.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C15_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/LE/C15_dyn_savefig.svg...


...

Done
C9_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/LE/C9_dyn_savefig.svg...


/tmp/ipykernel_47633/2587552383.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
C1_state_transition.csv
|-----------> plotting with basis key=X_umap
|-----------> skip filtering cluster by stack threshold when stacking color because it is not a numeric type
Saving figure to /syn1/liangzhen/jinhua_jilab_project/result/Figures/Figure3/transition_map/LE/C1_dyn_savefig.svg...


/tmp/ipykernel_47633/2587552383.py:14: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  adata_selected.obsm['X_umap'] = np.array(adata_selected.obsm['X_umap'])


Done
